In [ ]:
# Uncomment if packages are not yet installed
!pip -q install cmdstanpy pyarrow pandas numpy

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from cmdstanpy import CmdStanModel, install_cmdstan
from pathlib import Path

PARQUET_PATH = Path("/kaggle/input/datasets/karandeepshoker/full-dataset2/full_task_all_L1.parquet")
STAN_PATH    = Path("/kaggle/input/datasets/karandeepshoker/naive-models/model.stan")
VI_STAN_PATH = Path("/kaggle/input/datasets/karandeepshoker/naive-models/varInf.stan")  # adjust if needed

assert PARQUET_PATH.exists(), PARQUET_PATH
assert STAN_PATH.exists(), STAN_PATH
assert VI_STAN_PATH.exists(), VI_STAN_PATH
print("Found files OK.")

In [ ]:
# Install CmdStan only if not already installed
import cmdstanpy
try:
    cmdstanpy.cmdstan_path()
    print("CmdStan already installed at:", cmdstanpy.cmdstan_path())
except ValueError:
    install_cmdstan(version="2.35.0", cores=2)
    print("CmdStan installed.")

In [ ]:
# ── Read parquet metadata (K=9 for naive model) ───────────────────────────────
pf = pq.ParquetFile(str(PARQUET_PATH))
md = pf.metadata.metadata or {}

def md_get_str(key, default=None):
    b = md.get(key.encode("utf-8"))
    if b is None:
        return default
    try:
        return b.decode("utf-8")
    except Exception:
        return default

K_str     = md_get_str("K")
vocab_str = md_get_str("skill_vocab_json")

K = int(K_str) if K_str is not None else None
if K is None:
    raise RuntimeError("Missing Parquet metadata 'K'.")

skill_vocab = None
if vocab_str:
    try:
        skill_vocab = json.loads(vocab_str)
    except Exception:
        skill_vocab = eval(vocab_str, {"__builtins__": {}})

print("K =", K)   # should be 9
print("Skill vocabulary (Level-1):")
if skill_vocab:
    for k, v in sorted(skill_vocab.items(), key=lambda x: int(x[0])):
        print(f"  Skill {k:>2}: {v}")

In [ ]:
# ── Load and filter data ───────────────────────────────────────────────────────
df_raw = pd.read_parquet(PARQUET_PATH)

# Filter TimeTaken: must be present and in [5, 1800] seconds
df = df_raw.copy()
df = df[df["TimeTaken"].notna()]
df = df[(df["TimeTaken"] >= 5) & (df["TimeTaken"] <= 1800)].copy()

df["log_time"] = np.log(df["TimeTaken"].astype(float))
df["is_fast"]  = (df["TimeTaken"] < 20).astype(np.int32)

print("Rows after filter:", len(df))
df.head()

In [ ]:
# ── Student filtering ─────────────────────────────────────────────────────────
MIN_ANSWERS_PER_STUDENT = 50
N_STUDENTS  = 360
RANDOM_SEED = 42

counts = df.groupby("UserId").size()
eligible_users = counts[counts >= MIN_ANSWERS_PER_STUDENT].index
print(f"Eligible students (>= {MIN_ANSWERS_PER_STUDENT} answers): {len(eligible_users)}")

if len(eligible_users) < N_STUDENTS:
    raise ValueError(
        f"Only {len(eligible_users)} eligible students, cannot sample {N_STUDENTS}. "
        "Lower N_STUDENTS or MIN_ANSWERS_PER_STUDENT."
    )

rng = np.random.default_rng(RANDOM_SEED)
selected_users = rng.choice(eligible_users.to_numpy(), size=N_STUDENTS, replace=False)
df = df[df["UserId"].isin(selected_users)].copy()

post_counts = df.groupby("UserId").size()
print("Rows after student filter:",  len(df))
print("Unique students:",             df["UserId"].nunique())
print("Answers per student (min/median/mean/max):",
      int(post_counts.min()), float(post_counts.median()),
      float(post_counts.mean()), int(post_counts.max()))

In [ ]:
# ── Row sampling ──────────────────────────────────────────────────────────────
N_TARGET       = 25312
ANCHOR_PER_GROUP = 5

anchors = (
    df.groupby(["UserId", "IsCorrect"], group_keys=False)
      .apply(lambda g: g.sample(n=min(len(g), ANCHOR_PER_GROUP), random_state=RANDOM_SEED))
      .reset_index(drop=True)
)
remaining = df[~df["AnswerId"].isin(anchors["AnswerId"])]
need      = max(0, N_TARGET - len(anchors))
top_up    = remaining.sample(n=min(need, len(remaining)), random_state=RANDOM_SEED)

sample_df = pd.concat([anchors, top_up], ignore_index=True)
sample_df = sample_df.sample(n=min(N_TARGET, len(sample_df)), random_state=RANDOM_SEED).reset_index(drop=True)

print("Sample rows:",     len(sample_df))
print("Unique students:", sample_df["UserId"].nunique())
print("Unique questions:",sample_df["QuestionId"].nunique())
assert sample_df["UserId"].nunique() <= N_STUDENTS

In [ ]:
# ── Index remapping ───────────────────────────────────────────────────────────
sample_df["player_idx"]   = pd.factorize(sample_df["UserId"])[0].astype(np.int32) + 1
sample_df["question_idx"] = pd.factorize(sample_df["QuestionId"])[0].astype(np.int32) + 1

n_players   = int(sample_df["player_idx"].max())
n_questions = int(sample_df["question_idx"].max())

assert n_players   == sample_df["player_idx"].nunique()
assert n_questions == sample_df["question_idx"].nunique()

print("n_players:",   n_players)
print("n_questions:", n_questions)

In [ ]:
# ── Build sparse CSR skill structure ─────────────────────────────────────────
q_idx_map  = sample_df[["QuestionId","question_idx"]].drop_duplicates(subset=["QuestionId"])
q_skill_map = df_raw[["QuestionId","skill_indices"]].drop_duplicates(subset=["QuestionId"])
q_map = (
    q_idx_map
    .merge(q_skill_map, on="QuestionId", how="left")
    .sort_values("question_idx")
    .reset_index(drop=True)
)
assert len(q_map) == n_questions

def normalize_skill_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]
    if isinstance(x, (list, tuple)):
        return [int(v) for v in x]
    try:
        return [int(v) for v in list(x)]
    except Exception:
        return []

skill_lists = [normalize_skill_list(x) for x in q_map["skill_indices"]]

# CSR arrays (1-indexed for Stan)
skill_flat  = np.array([s for sl in skill_lists for s in sl], dtype=np.int32)
skill_count = np.array([len(sl) for sl in skill_lists], dtype=np.int32)
skill_start = np.zeros(n_questions, dtype=np.int32)
pos = 1
for i, cnt in enumerate(skill_count):
    skill_start[i] = pos
    pos += cnt

n_skill_entries = int(skill_flat.shape[0])
print("n_skill_entries:", n_skill_entries)
print(f"Skill sparsity: {1 - n_skill_entries / (n_questions * K):.3f}")

In [ ]:
# ── Player covariates ─────────────────────────────────────────────────────────
player_meta = (
    sample_df.groupby("player_idx", sort=True)
    .agg(Age=("Age","first"), Gender=("Gender","first"))
    .reset_index()
)
assert len(player_meta) == n_players

age_raw  = player_meta["Age"].astype(float).to_numpy()
age_mean = np.nanmean(age_raw)
age_sd   = np.nanstd(age_raw)
if not np.isfinite(age_sd) or age_sd == 0:
    age_sd = 1.0
age_z = np.where(np.isfinite(age_raw), (age_raw - age_mean) / age_sd, 0.0).astype(np.float64)

gender = player_meta["Gender"].to_numpy()
gender = np.where(pd.isna(gender), 0, gender).astype(np.int32)
gender = np.clip(gender, 0, 2)

print("age_z shape:",      age_z.shape)
print("gender counts:",    pd.Series(gender).value_counts().to_dict())

In [ ]:
# ── Assemble Stan data dict ───────────────────────────────────────────────────
stan_data = {
    "N":            int(len(sample_df)),
    "n_players":    n_players,
    "n_questions":  n_questions,
    "n_skills":     int(K),   # 9 for the naive model

    "player_id":    sample_df["player_idx"].astype(np.int32).to_numpy(),
    "question_id":  sample_df["question_idx"].astype(np.int32).to_numpy(),
    "response":     sample_df["IsCorrect"].astype(np.int32).to_numpy(),
    "is_fast":      sample_df["is_fast"].astype(np.int32).to_numpy(),
    "log_time":     sample_df["log_time"].astype(np.float64).to_numpy(),

    "n_skill_entries": int(n_skill_entries),
    "skill_flat":   skill_flat.astype(np.int32),
    "skill_start":  skill_start.astype(np.int32),
    "skill_count":  skill_count.astype(np.int32),

    "age_z":        age_z.astype(np.float64),
    "gender":       gender.astype(np.int32),
}

assert stan_data["player_id"].max()   == stan_data["n_players"]
assert stan_data["question_id"].max() == stan_data["n_questions"]
assert len(stan_data["age_z"])        == stan_data["n_players"]
assert len(stan_data["skill_start"])  == stan_data["n_questions"]
print("stan_data assembled OK.")
print(f"N={stan_data['N']:,}  n_players={n_players}  n_questions={n_questions}  K={K}")

In [ ]:
# Kaggle input datasets are read-only. CmdStan writes .hpp/exe next to the .stan file,
# so we must copy the Stan files to a writable directory first.
from shutil import copy2

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

STAN_LOCAL_PATH = WORK_DIR / "main.stan"
copy2(STAN_PATH, STAN_LOCAL_PATH)

VI_LOCAL_PATH = None
if VI_STAN_PATH is not None and Path(VI_STAN_PATH).exists():
    VI_LOCAL_PATH = WORK_DIR / Path(VI_STAN_PATH).name
    copy2(VI_STAN_PATH, VI_LOCAL_PATH)

print("Copied Stan files to:")
print(" -", STAN_LOCAL_PATH)
if VI_LOCAL_PATH:
    print(" -", VI_LOCAL_PATH)

model = CmdStanModel(stan_file=str(STAN_LOCAL_PATH))
print("Compiled main.stan OK.")

In [ ]:
fit = model.sample(
    data=stan_data,
    chains=2,
    parallel_chains=2,
    iter_warmup=300,
    iter_sampling=300,
    show_progress=True,
)

In [ ]:
import pandas as pd
import os

# Create a safe directory for your outputs
output_dir = "/kaggle/working/bayesian_results"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------------
# 1. THE ULTIMATE BACKUP (Raw Stan CSVs)
# ---------------------------------------------------------
# This copies the gigabytes of raw MCMC data from the hidden /tmp/ folder 
# to your visible Kaggle working directory. If everything else fails, 
# you can rebuild your fit from these files.
fit.save_csvfiles(dir=output_dir)

# ---------------------------------------------------------
# 2. THE DIAGNOSTICS TABLE (For the Appendix & Convergence Check)
# ---------------------------------------------------------
summary_df = fit.summary()
summary_df.to_csv(f"{output_dir}/nuts_mcmc_summary.csv")

# ---------------------------------------------------------
# 3. LOG-LIKELIHOOD DRAWS (For LOO-CV & Model Comparison)
# ---------------------------------------------------------
# You MUST have this to compare your Complex Model to your Naive Model
log_lik_df = fit.draws_pd(vars=["log_lik"])
log_lik_df.to_csv(f"{output_dir}/log_lik_draws.csv", index=False)

# ---------------------------------------------------------
# 4. POSTERIOR PREDICTIVE CHECKS (y_rep)
# ---------------------------------------------------------
# You did the mean calculation, but you might want to plot a histogram 
# of the accuracy distributions for your "Critical Evaluation" section.
y_rep_df = fit.draws_pd(vars=["y_rep"])
y_rep_df.to_csv(f"{output_dir}/y_rep_draws.csv", index=False)

# ---------------------------------------------------------
# 5. TARGET STUDENT PROFILES (For the "Real World" Inference Task)
# ---------------------------------------------------------
# Grab the 69 skill posteriors for the first 3 students so you can 
# make the "Skill Profile" bar charts the rubric asks for.
# Get all user_skill draws then filter to first 3 students' columns
all_skill_df = fit.draws_pd(vars=["user_skill"])
student_cols = [c for c in all_skill_df.columns
                if any(c.startswith(f"user_skill[{i},") for i in range(1, 4))]
student_df = all_skill_df[student_cols]
student_df.to_csv(f"{output_dir}/target_student_profiles.csv", index=False)

print(f"All critical data saved successfully to {output_dir}!")

In [ ]:
# Extract a small set of draws for y_rep
draws = fit.draws_pd(vars=["y_rep"])
print("y_rep draws shape:", draws.shape)

obs_mean = float(np.mean(stan_data["response"]))
# draws_pd returns wide columns y_rep[1], y_rep[2], ...
yrep_cols = [c for c in draws.columns if c.startswith("y_rep[")]
pp_mean = float(draws[yrep_cols].to_numpy().mean())

print("Observed mean correct:", round(obs_mean, 4))
print("Posterior predictive mean correct (across draws/obs):", round(pp_mean, 4))

In [ ]:
from cmdstanpy import CmdStanModel

if VI_LOCAL_PATH is None:
    raise FileNotFoundError("VI_STAN_PATH not found (check filename/casing or dataset path).")

vi_model = CmdStanModel(stan_file=str(VI_LOCAL_PATH))
print("Compiled VarInf.stan OK.")

In [ ]:
vi_fit = vi_model.variational(
    data=stan_data,
    algorithm="meanfield",     # or "fullrank" (slower, more flexible)
    iter=20000,
    grad_samples=2,
    elbo_samples=100,
    draws=800,       # posterior draws from the variational approx
)
print("VI done.")

In [ ]:
# ---------------------------------------------------------
# 1. THE RAW VI CSV FILES (Backup)
# ---------------------------------------------------------
vi_fit.save_csvfiles(dir=output_dir)

# ---------------------------------------------------------
# 2. THE FULL VI DRAWS (For Density Plots)
# ---------------------------------------------------------
# Build draws DataFrame via stan_variable (vi_fit.sample removed in this cmdstanpy version)
global_vars = ["gamma_base", "alpha", "beta_time", "mu_log_time", "sigma_log_time"]
vi_draws_df = pd.DataFrame({v: vi_fit.stan_variable(v, mean=False) for v in global_vars})
vi_draws_df.to_csv(f"{output_dir}/vi_draws_full.csv", index=False)

# ---------------------------------------------------------
# 3. THE VI SUMMARY (For the Comparison Table)
# ---------------------------------------------------------
vi_summary_df = vi_draws_df.describe()
vi_summary_df.to_csv(f"{output_dir}/vi_summary_comparison.csv")

print(f"VI data successfully saved to {output_dir}!")

In [ ]:
import pandas as pd

global_vars = ["gamma_base", "alpha", "beta_time", "mu_log_time", "sigma_log_time"]
draws = pd.DataFrame({v: vi_fit.stan_variable(v, mean=False) for v in global_vars})
draws.describe()

In [ ]:
vi_fit.runset.csv_files  # list of output csv paths